# Startup/Product Forecasting

Generate a forecasting dataset about startup survival using YC company data from [yc-oss/api](https://github.com/yc-oss/api). Questions focus on longevity (will this company still exist in X years?) and funding (will they reach Series A/B?). WebSearchLabeler verifies outcomes via web search. Seeds are stratified by outcome (Inactive vs Acquired/Public/Active) to reduce positive bias.

In [1]:
%pip install python-dotenv pandas
%pip install -e ..

from IPython.display import clear_output
clear_output()

import pandas as pd
from dotenv import load_dotenv

load_dotenv()

True

## Set up the client

Sign up at [dashboard.lightningrod.ai](https://dashboard.lightningrod.ai/?redirect=/api) to get your API key and **$50 of free credits**.

In [2]:
from lightningrod import LightningRod
from lightningrod.utils import config

api_key = config.get_config_value("LIGHTNINGROD_API_KEY")
lr = LightningRod(api_key=api_key)

## YC company data

Fetch companies from [yc-oss/api](https://github.com/yc-oss/api). Filter by `launched_at` (2015–2022) so questions like "Will X still be around in 3 years?" have resolution dates in the past. Stratify by `status` (Inactive vs Acquired/Public/Active) for ~50/50 split to reduce positive bias.

In [ ]:
import random
from datetime import datetime
from urllib.request import urlopen
import json

from lightningrod import create_sample

YC_API_URL = "https://yc-oss.github.io/api/companies/all.json"
# prefer older batches: they have more resolved outcomes
LAUNCH_START = int(datetime(2015, 1, 1).timestamp())
LAUNCH_END = int(datetime(2022, 1, 1).timestamp())
MIN_DESC_LENGTH = 100  # skip companies with little context for question generation
SAMPLES_PER_STATUS_GROUP = 175  # stratified sampling: ~50/50 failed vs successful to reduce positive bias

with urlopen(YC_API_URL) as resp:
    companies = json.load(resp)

def _build_seed_text(c):
    # exclude status: it would leak the outcome (Inactive vs Acquired) and trivialize the forecasting task
    desc = (c.get("long_description") or "").strip() or (c.get("one_liner") or "")
    tags = ", ".join(c.get("tags") or [])
    return (
        f"Title: {c.get('name', '')}\n"
        f"One-liner: {c.get('one_liner', '')}\n\n"
        f"Description: {desc}\n\n"
        f"Website: {c.get('website', '')}\n"
        f"YC URL: {c.get('url', '')}\n"
        f"Batch: {c.get('batch', '')}\n"
        f"Industry: {c.get('industry', '')}\n"
        f"Tags: {tags}"
    )

# launched_at range: older batches have more resolved outcomes; MIN_DESC_LENGTH ensures usable context
filtered = [
    c for c in companies
    if (c.get("launched_at") and LAUNCH_START <= c["launched_at"] < LAUNCH_END
        and len((c.get("long_description") or "") or (c.get("one_liner") or "")) >= MIN_DESC_LENGTH)
]

# stratify by outcome so input has ~50/50 failed vs successful; raw YC data skews toward success
inactive = [c for c in filtered if c.get("status") == "Inactive"]
successful = [c for c in filtered if c.get("status") in ("Acquired", "Public", "Active")]

random.seed(42)
n_inactive = min(len(inactive), SAMPLES_PER_STATUS_GROUP)
n_successful = min(len(successful), SAMPLES_PER_STATUS_GROUP)
sampled_inactive = random.sample(inactive, n_inactive)
sampled_successful = random.sample(successful, n_successful)
selected = sampled_inactive + sampled_successful
random.shuffle(selected)

samples = []
for c in selected:
    seed_text = _build_seed_text(c)
    seed_date = datetime.utcfromtimestamp(c["launched_at"]) if c.get("launched_at") else None
    samples.append(create_sample(seed_text, seed_date=seed_date))

input_dataset = lr.datasets.create_from_samples(samples, batch_size=1000)
print(f"Created input dataset: {input_dataset.id} ({len(samples)} companies: {n_inactive} Inactive, {n_successful} Acquired/Public/Active)")

Created input dataset: bfbbc6b6-df33-412b-a931-7927cb1573e1 (350 companies: 175 Inactive, 175 Acquired/Public/Active)


## Build the pipeline

ForwardLookingQuestionGenerator produces questions with prediction_date and date_close. WebSearchLabeler verifies survival via web search. No NewsContextGenerator — the seed contains the full company info; use `drop_missing_context=False` in prepare.

In [4]:
INSTRUCTIONS = """
Generate binary forecasting questions about whether this YC-backed startup will survive or succeed.
Focus on: (1) longevity — will the company still exist in X years? (2) funding — will they reach the next stage?
Use the company name, description, website, and YC URL to identify the startup. Questions must be forward-looking from the company's launch date and verifiable via web search.
"""

EXAMPLES = [
    "Will this company still be operational in 3 years?",
    "Will this startup still be around in 2 years?",
    "Will this company raise Series A within 18 months?",
]

BAD_EXAMPLES = [
    "What technology does this use?",
    "When was this founded?",
    "Is this B2B or B2C?",
]

In [ ]:
from lightningrod import (
    BinaryAnswerType,
    ForwardLookingQuestionGenerator,
    WebSearchLabeler,
    QuestionRenderer,
    QuestionPipeline,
)

answer_type = BinaryAnswerType()

pipeline = QuestionPipeline(
    question_generator=ForwardLookingQuestionGenerator(
        instructions=INSTRUCTIONS,
        examples=EXAMPLES,
        bad_examples=BAD_EXAMPLES,
        answer_type=answer_type,
        questions_per_seed=2,
    ),
    labeler=WebSearchLabeler(
        answer_type=answer_type,
        confidence_threshold=0.5,
    ),
    renderer=QuestionRenderer(answer_type=answer_type),
)

> Note: Processing can take several minutes (question generation, web search labeling).

## Run the pipeline

In [12]:
dataset = lr.transforms.run(
    pipeline,
    input_dataset=input_dataset,  # YC seeds from create_from_samples; no seed_generator
    max_questions=500,
    name="Startup forecasting",
)
samples = dataset.download()

pct = (sum(1 for s in samples if s.is_valid is True) / len(samples) * 100) if samples else 0
print(f"{len(samples)} samples ({pct:.1f}% valid)")

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│  >> Warning                                                                                                     │
│                                                                                                                 │
│  Estimated cost ($78.52) exceeds current balance ($49.76). Consider adding credits before running this job.     │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Output()

1060 samples (19.0% valid)


## Prepare the dataset

Filter valid samples, deduplicate, and split into train/test. Use `drop_missing_context=False` (no NewsContextGenerator) and `days_to_resolution_range=(365, None)` to keep 1+ year horizons.

In [17]:
from lightningrod.training import prepare_for_training

train, test = prepare_for_training(
    samples,
    answer_type,
    test_size=0.2,
    split_strategy="temporal",
    include_assistant=True,
    filter_leaky_train=False,
    days_to_resolution_range=(365, None),  # keep only questions with 1+ year horizon (longevity/funding)
    drop_missing_context=False,  # no NewsContextGenerator; seed has full company info
)

for name, data in [("Train", train), ("Test", test)]:
    if data:
        yes_count = sum(s["label"] or 0 for s in data)
        print(f"{name}: {len(data)} rows, {yes_count/len(data)*100:.1f}% yes")
    else:
        print(f"{name}: 0 rows")

Train: 153 rows, 48.4% yes
Test: 39 rows, 53.8% yes


## Results

In [14]:
def _display_head(data, name, n=5):
    if not data:
        print(f"{name}: no rows")
        return
    df = pd.DataFrame(data[:n])
    # Prevent question_text truncation in pandas DataFrame display
    pd.set_option('display.max_colwidth', None)
    cols = ["question_text", "prediction_date", "date_close", "resolution_date", "label", "label_confidence"]
    display_cols = [c for c in cols if c in df.columns]
    print(f"{name} (head):")
    display(df[display_cols])

_display_head(train, "Train")
_display_head(test, "Test")

Train (head):


,question_text,prediction_date,date_close,resolution_date,label,label_confidence
0,"Will WebsiteVoice, LLC or its parent company have officially announced a venture capital funding round (including Seed, Series A, or later) of at least $1 million USD by July 1, 2022?",2019-01-07T02:56:45,2022-07-01T00:00:00,2022-07-01T00:00:00,0.0,0.9
1,"Will the company or project behind Polar (getpolarized.io) announce a venture capital funding round (Seed, Series A, or later) of at least $1 million USD by January 1, 2022?",2019-01-09T15:43:35,2022-01-01T00:00:00,2022-01-01T00:00:00,0.0,1.0
2,"Will Mikado Software, the creator of the 'workstation' project, be listed as an active company on the UK Companies House register as of January 1, 2025?",2019-01-10T22:52:58,2025-01-05T00:00:00,2025-01-01T00:00:00,1.0,1.0
3,"Will the website heyfromthefuture.com still be operational on July 14, 2025?",2019-01-14T12:48:00,2025-07-14T00:00:00,2024-12-31T00:00:00,0.0,1.0
4,"Will the 'Plain Freelance Contract' project, as launched at plainfreelancecontract.com, still be online and publicly accessible on January 16, 2024?",2019-01-15T14:32:50,2024-01-16T00:00:00,2024-01-16T00:00:00,1.0,1.0


Test (head):


,question_text,prediction_date,date_close,resolution_date,label,label_confidence
0,"Will the website Screentop.gg be operational and accessible for play on January 10, 2023?",2020-01-10T21:24:06,2023-01-10T00:00:00,2023-01-10T00:00:00,1.0,1.0
1,"Will the domain podnami.com resolve to an active, operational website dedicated to technology podcast discovery on January 12, 2023?",2020-01-12T16:35:58,2023-01-12T00:00:00,2023-01-12T00:00:00,0.0,0.9
2,"Will the Diary Email service (diaryemail.com) still be operational as a live website on January 14, 2023?",2020-01-14T14:37:40,2023-01-14T00:00:00,2023-01-14T00:00:00,1.0,0.9
3,"Will vesoft-inc, the company behind Nebula Graph, announce a Series B funding round by December 31, 2022?",2020-01-15T02:27:06,2022-12-31T00:00:00,2022-12-31T00:00:00,0.0,1.0
4,"Will the 'Screenshot Hero' app by Asad Memon be available for download on the official Apple App Store on January 1, 2023?",2020-01-16T15:24:38,2023-01-01T00:00:00,2023-01-01T00:00:00,0.0,0.9


## Uploading the dataset to HuggingFace

Once we have a training-ready dataset, we can push it to Hugging Face for sharing or downstream use.

In [15]:
%pip install datasets -q

from datasets import Dataset, DatasetDict
from lightningrod.utils import config

dataset = DatasetDict({
    "train": Dataset.from_list(train),
    "test": Dataset.from_list(test),
})
print(f"Train: {len(dataset['train'])} rows, Test: {len(dataset['test'])} rows")
print("Columns:", dataset["train"].column_names[:8], "...")

DATASET_PATH = f"{config.get_config_value('HF_USERNAME')}/startup-forecasting-demo"
dataset.push_to_hub(DATASET_PATH, token=config.get_config_value("HF_ACCESS_TOKEN"))


[notice] A new release of pip available: 22.3.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
Train: 153 rows, Test: 39 rows
Columns: ['question_text', 'date_close', 'event_date', 'resolution_criteria', 'prediction_date', 'label', 'answer_type', 'label_confidence'] ...


Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

...

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

README.md:   0%|          | 0.00/933 [00:00<?, ?B/s]

CommitInfo(commit_url='https://huggingface.co/datasets/bart/startup-forecasting-demo/commit/2678df1d6d456022b728a9e086896fc0301a34dd', commit_message='Upload dataset', commit_description='', oid='2678df1d6d456022b728a9e086896fc0301a34dd', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/bart/startup-forecasting-demo', endpoint='https://huggingface.co', repo_type='dataset', repo_id='bart/startup-forecasting-demo'), pr_revision=None, pr_num=None)